# 54 — Evidence Cube Builder (timezone-safe replacement)

This replacement fixes the full-pipeline failure caused by mixing
timezone-aware and timezone-naive timestamps when comparing window dates.

Main change:
- all parsed dates are coerced to UTC
- timezone is removed
- dates are normalized before comparison

In [ ]:
import os, json, hashlib, platform, sys
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yaml

PROJECT_ROOT = Path.cwd().resolve().parents[0] if (Path.cwd().name == "notebooks") else Path.cwd().resolve()
OUTPUTS = PROJECT_ROOT / "outputs"
CONFIGS = PROJECT_ROOT / "configs"
OUTPUTS.mkdir(exist_ok=True)

def utcnow():
    return datetime.now(timezone.utc).isoformat()

def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024*1024), b""):
            h.update(chunk)
    return h.hexdigest()

def write_json(p: Path, obj):
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def load_json(p: Path):
    return json.loads(p.read_text(encoding="utf-8"))

def load_yaml(p: Path):
    return yaml.safe_load(p.read_text(encoding="utf-8")) if p.exists() else {}

def safe_read_csv(path_str):
    p = Path(path_str)
    if not p.is_absolute():
        p = PROJECT_ROOT / p
    if not p.exists():
        return pd.DataFrame(), {"path": str(p), "status": "missing"}
    try:
        if p.stat().st_size == 0:
            return pd.DataFrame(), {"path": str(p), "status": "empty_file"}
        df = pd.read_csv(p)
        if df.empty:
            return pd.DataFrame(), {"path": str(p), "sha256": sha256_file(p), "status": "no_rows"}
        return df, {"path": str(p), "sha256": sha256_file(p), "status": "ok"}
    except pd.errors.EmptyDataError:
        return pd.DataFrame(), {"path": str(p), "status": "empty_data"}
    except Exception as e:
        return pd.DataFrame(), {"path": str(p), "status": f"error:{e}"}

def manifest_base(step_name: str, config_paths=None):
    return {
        "project": "AirQuality26",
        "step": step_name,
        "created_at_utc": utcnow(),
        "platform": {
            "python": sys.version,
            "platform": platform.platform(),
        },
        "configs": {str(p): sha256_file(Path(p)) for p in (config_paths or []) if Path(p).exists()},
        "artifacts": [],
        "notes": []
    }

def _to_naive_normalized(series):
    ts = pd.to_datetime(series, errors="coerce", utc=True)
    try:
        ts = ts.dt.tz_convert(None)
    except Exception:
        pass
    return ts.dt.normalize()

def normalise_date_col(df):
    for c in df.columns:
        if c.lower() in {"date","day","datetime","timestamp","time","time_utc","publishedat","published_at"}:
            out = df.copy()
            out["date"] = _to_naive_normalized(out[c])
            return out, c
    return df.copy(), None

run_cfg = load_yaml(CONFIGS / "run.yml")
phase_cfg = load_yaml(CONFIGS / "phase50.yml")
date_from = run_cfg.get("run", {}).get("date_from", "2025-01-01")
date_to = run_cfg.get("run", {}).get("date_to", "2025-01-31")

In [ ]:
step = "54_evidence_cube_builder"
out = OUTPUTS / step
out.mkdir(parents=True, exist_ok=True)

dates = pd.date_range(date_from, date_to, freq="D").tz_localize(None)
daily = pd.DataFrame({"date": dates})

fusion_candidates = [
    OUTPUTS / "32_newhaven_weather_ground_news_fusion" / "fusion_daily.csv",
    OUTPUTS / "newhaven_fusion" / "fusion_daily.csv",
]
fusion_df = pd.DataFrame()
chosen_fusion = None
for p in fusion_candidates:
    df, meta = safe_read_csv(p)
    if not df.empty:
        df, _ = normalise_date_col(df)
        fusion_df = df.copy()
        chosen_fusion = str(p.relative_to(PROJECT_ROOT))
        break

if not fusion_df.empty and "date" in fusion_df.columns:
    keep_cols = [c for c in fusion_df.columns if c in {
        "date","fusion_score","ground_rows","article_count","mean_wind_speed_ms",
        "mean_cloud_cover_pct","mean_temperature_c"
    }]
    daily = daily.merge(fusion_df[keep_cols], on="date", how="left")

news_kept, _ = safe_read_csv("outputs/53_news_relevance_filtering/news_relevance_kept.csv")
if not news_kept.empty:
    news_kept["date"] = _to_naive_normalized(news_kept["published_at"])
    news_daily = news_kept.groupby("date", as_index=False).agg(
        relevant_news_count=("title","size"),
        relevant_headlines=("title", lambda x: " | ".join(list(map(str, list(x)[:3]))))
    )
    daily = daily.merge(news_daily, on="date", how="left")

phase40, _ = safe_read_csv("outputs/43_window_ranking/window_top_candidates.csv")
daily["phase40_window_flag"] = 0
phase40_window_count = 0
if not phase40.empty:
    phase40["window_start"] = _to_naive_normalized(phase40["window_start"])
    phase40["window_end"] = _to_naive_normalized(phase40["window_end"])
    phase40 = phase40.dropna(subset=["window_start", "window_end"]).copy()
    phase40_window_count = len(phase40)
    for _, row in phase40.iterrows():
        mask = (daily["date"] >= row["window_start"]) & (daily["date"] <= row["window_end"])
        daily.loc[mask, "phase40_window_flag"] = 1

legacy, _ = safe_read_csv("outputs/34_newhaven_target_window_selector/window_top_candidates.csv")
daily["legacy_window_flag"] = 0
legacy_window_count = 0
if not legacy.empty:
    for c in ["window_start","start","window_end","end"]:
        if c in legacy.columns:
            legacy[c] = _to_naive_normalized(legacy[c])
    sc = "window_start" if "window_start" in legacy.columns else ("start" if "start" in legacy.columns else None)
    ec = "window_end" if "window_end" in legacy.columns else ("end" if "end" in legacy.columns else None)
    if sc and ec:
        legacy = legacy.dropna(subset=[sc, ec]).copy()
        legacy_window_count = len(legacy)
        for _, row in legacy.iterrows():
            mask = (daily["date"] >= row[sc]) & (daily["date"] <= row[ec])
            daily.loc[mask, "legacy_window_flag"] = 1

station_summary = load_json(OUTPUTS / "51_station_provenance_validation" / "station_validation_summary.json") if (OUTPUTS / "51_station_provenance_validation" / "station_validation_summary.json").exists() else {}
weather_summary = load_json(OUTPUTS / "52_weather_validation_and_fallback_audit" / "weather_validation_summary.json") if (OUTPUTS / "52_weather_validation_and_fallback_audit" / "weather_validation_summary.json").exists() else {}
news_summary = load_json(OUTPUTS / "53_news_relevance_filtering" / "news_relevance_summary.json") if (OUTPUTS / "53_news_relevance_filtering" / "news_relevance_summary.json").exists() else {}

daily["station_aliasing_flag"] = 1 if station_summary.get("sites_flagged_aliasing", 0) > 0 else 0
daily["station_distance_flag"] = 1 if station_summary.get("sites_flagged_distance", 0) > 0 else 0
daily["weather_ready_flag"] = 1 if weather_summary.get("weather_ready", False) else 0
daily["news_ready_flag"] = 1 if news_summary.get("kept_article_count", 0) > 0 else 0

for c in ["fusion_score","ground_rows","article_count","relevant_news_count"]:
    if c not in daily.columns:
        daily[c] = 0
    daily[c] = pd.to_numeric(daily[c], errors="coerce").fillna(0)

daily["evidence_score"] = (
    daily["fusion_score"] * 0.35
    + daily["ground_rows"] * 0.15
    + daily["relevant_news_count"] * 0.15
    + daily["phase40_window_flag"] * 0.20
    + daily["legacy_window_flag"] * 0.15
)

daily.to_csv(out / "evidence_cube_daily.csv", index=False)

summary = {
    "days": int(len(daily)),
    "max_evidence_score": float(daily["evidence_score"].max()) if len(daily) else 0.0,
    "weather_ready": bool(daily["weather_ready_flag"].max()) if len(daily) else False,
    "chosen_fusion_source": chosen_fusion,
    "phase40_window_count": int(phase40_window_count),
    "legacy_window_count": int(legacy_window_count),
}
write_json(out / "evidence_cube_summary.json", summary)

manifest = manifest_base(step, [CONFIGS / "run.yml", CONFIGS / "phase50.yml"])
for p in [out / "evidence_cube_daily.csv", out / "evidence_cube_summary.json"]:
    manifest["artifacts"].append({"path": str(p), "sha256": sha256_file(p)})
manifest["summary"] = summary
write_json(out / "manifest.json", manifest)

display(daily.head(20))
print(summary)